# Chapter 16: Understanding Queries and Routing

Transparent deterministic rules on query shapes; route labels are not evidence outcomes.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
if str(ROOT / "examples") not in sys.path:
    sys.path.insert(0, str(ROOT / "examples"))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:
from document_identity import Collection, digest_bytes
from evidence_units import evidence_success, version_key
from query_retrieval import Retrieval, interpret, reformulate, Budget
import json
from pathlib import Path

fixture = json.loads((ROOT / "experiments" / "part4-complex" / "development-corpus.json").read_text())
relations = json.loads((ROOT / "experiments" / "part4-complex" / "development-relations.json").read_text())
config = json.loads((ROOT / "experiments" / "part4-complex" / "config.json").read_text())
needs = json.loads((ROOT / "experiments" / "part4-complex" / "development-queries.json").read_text())["queries"]

def engine(view="main"):
    return Retrieval(Collection(fixture), view, relations, config)

# Show the recorded route and reason for every development request.
for q in needs:
    p = engine().plan(q["query"], q["query_id"])
    print(f"{q['query_id']}: route={p['route']!r:12} reason={p['route_reason']!r} intent={p['interpretation']['intent']!r} entities={p['interpretation']['entities']} identifiers={p['interpretation']['identifiers']}")


In [ ]:
# Rule precedence: a contact relationship overrides a project identifier inside the same request.
p = engine().plan("Which contact serves project PR-21?", "q")
assert p["route"] == "structured"
assert "PR-21" in p["interpretation"]["identifiers"]
print("relation route:", p["route"], p["route_reason"])

# Ambiguous public alias/acronym stays unresolved instead of choosing one entity.
p = engine().plan("Which contact serves Rowan?", "q")
assert p["route"] == "abstain"
assert len(p["interpretation"]["entities"]) == 2
print("ambiguous route:", p["route"], p["interpretation"]["entities"])

# A longer qualified alias resolves to one entity without collapsing the shared display name.
p = engine().plan("Which contact serves Rowan north?", "q")
assert p["interpretation"]["entities"] == ["org:development:north"]
print("qualified alias:", p["interpretation"]["entities"])


In [ ]:
# Route label != successful answer: run the policy route and an always-lexical override, then compare bundle coverage.
q = next(x for x in needs if x["query_id"] == "development-08")
e = engine()
p = e.plan(q["query"], q["query_id"])
policy = e.execute(p, "policy")
lexical = e.execute(p, "lexical")
print("policy route:", policy["executed_route"], "status:", policy["status"], "chars:", policy["selection"]["characters"])
print("lexical override status:", lexical["status"], "chars:", lexical["selection"]["characters"])
print("policy bundle:", evidence_success(q, [c["unit"] for c in policy["selection"]["selected"]]))
print("lexical bundle:", evidence_success(q, [c["unit"] for c in lexical["selection"]["selected"]]))
assert policy["executed_route"] == "structured"


In [ ]:
# An ambiguous acronym produces no candidates; the route is a decision, not an answer.
p = engine().plan("DR window", "q")
out = engine().execute(p, "policy")
print("route:", p["route"], "execution status:", out["status"], "selected:", len(out["selection"]["selected"]))
assert p["route"] == "abstain"
assert out["status"] == "unresolved_request"
assert out["selection"]["selected"] == []


## Observed versus not established
Observed: deterministic route labels, precedence, abstention, and the distinction between a route decision and evidence coverage. Not established: calibrated intent confidence, general router accuracy, or downstream answer success.